# dask_image

## Example 1: a Gaussian filter over chunks matches SciPy

In [ ]:
# Example 1: ndfilters.gaussian_filter adds the halo each chunk needs, so the result equals the in-memory filter
import dask.array as da
import numpy as np
import scipy.ndimage as ndi
from dask_image import ndfilters

image = np.random.default_rng(0).random((64, 64))
chunked = da.from_array(image, chunks=(16, 16))
smoothed = ndfilters.gaussian_filter(chunked, sigma=2)
assert smoothed.chunks == chunked.chunks
assert np.allclose(smoothed.compute(), ndi.gaussian_filter(image, sigma=2))
print("Example 1:", len(chunked.chunks[0]) * len(chunked.chunks[1]), "chunks, matches scipy.ndimage")

## Example 2: label connected regions and measure them

In [ ]:
# Example 2: ndmeasure.label numbers regions across chunk borders; ndmeasure.area sums each label's pixels
import dask.array as da
import numpy as np
from dask_image import ndmeasure

mask = np.zeros((20, 20), dtype=bool)
mask[2:6, 2:6] = True; mask[8:18, 9:11] = True; mask[15:19, 15:19] = True
labels, count = ndmeasure.label(da.from_array(mask, chunks=10))
count = int(count.compute()) if hasattr(count, "compute") else int(count)
areas = ndmeasure.area(da.ones_like(labels), labels, np.arange(1, count + 1)).compute()
assert count == 3 and sorted(areas.tolist()) == [16, 16, 20], (count, areas)
print("Example 2:", count, "regions with areas", sorted(areas.tolist()))

## Example 3: binary morphology

In [ ]:
# Example 3: ndmorph.binary_dilation / binary_erosion grow and shrink a square, same as SciPy
import dask.array as da
import numpy as np
import scipy.ndimage as ndi
from dask_image import ndmorph

square = np.zeros((16, 16), dtype=bool); square[5:11, 5:11] = True
grown = ndmorph.binary_dilation(da.from_array(square, chunks=8), iterations=1).compute()
shrunk = ndmorph.binary_erosion(da.from_array(square, chunks=8), iterations=1).compute()
assert np.array_equal(grown, ndi.binary_dilation(square)) and np.array_equal(shrunk, ndi.binary_erosion(square))
assert (square.sum(), grown.sum(), shrunk.sum()) == (36, 60, 16)
print("Example 3: pixels", square.sum(), "-> dilated", grown.sum(), "/ eroded", shrunk.sum())

## Example 4: read a TIFF stack lazily

In [ ]:
# Example 4: dask_image.imread.imread opens the file through PIMS and gives one chunk per frame
import pathlib, tempfile
import numpy as np
import tifffile
from dask_image.imread import imread

path = pathlib.Path(tempfile.mkdtemp()) / "stack.tif"
stack = np.arange(5 * 6 * 7, dtype=np.uint16).reshape(5, 6, 7)
tifffile.imwrite(path, stack, photometric="minisblack")
lazy = imread(str(path))
assert lazy.shape == (5, 6, 7) and lazy.chunksize[0] == 1
assert np.array_equal(lazy[2:4].compute(), stack[2:4])
print("Example 4:", lazy.shape, "chunks", lazy.chunksize, lazy.dtype)

## Example 5: a Fourier-domain shift

In [ ]:
# Example 5: ndfourier.fourier_shift on chunks equals scipy.ndimage.fourier_shift
import dask.array as da
import numpy as np
import scipy.ndimage as ndi
from dask_image import ndfourier

image = np.random.default_rng(1).random((32, 32))
spectrum = np.fft.fft2(image)
shifted = ndfourier.fourier_shift(da.from_array(spectrum, chunks=16), shift=(3, -2)).compute()
assert np.allclose(shifted, ndi.fourier_shift(spectrum, shift=(3, -2)))
moved = np.fft.ifft2(shifted).real
assert np.allclose(moved, np.roll(image, (3, -2), axis=(0, 1)))
print("Example 5: shift by (3, -2) matches np.roll:", bool(np.allclose(moved, np.roll(image, (3, -2), axis=(0, 1)))))